# 01 - Data Loading & Pre-flight
Loads all raw CRO files, verifies required columns, checks null rates and date ranges.
**No features computed here.** NB02 does all feature engineering.

**Inputs:** data/raw/01_CRO_Raw/
**Outputs:** None (read-only checks)

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from src.config import RAW_FILES, FEATURE_COLS, TRAIN_CUTOFF_DATE, TEST_CUTOFF_DATE
print(f"Feature count : {len(FEATURE_COLS)}")
print(f"Train cutoff  : {TRAIN_CUTOFF_DATE}")
print(f"Test cutoff   : {TEST_CUTOFF_DATE}")

Feature count : 84
Train cutoff  : 2022-12-31
Test cutoff   : 2023-12-31


In [2]:
print("Loading raw data...")
companies = pd.read_csv(RAW_FILES["company_records"], low_memory=False)
fs22      = pd.read_csv(RAW_FILES["fs_2022"],         low_memory=False)
fs23      = pd.read_csv(RAW_FILES["fs_2023"],         low_memory=False)
fs24      = pd.read_csv(RAW_FILES["fs_2024"],         low_memory=False)
diss      = pd.read_csv(RAW_FILES["dissolutions"],    low_memory=False)

# Normalise column names
for df in [companies, fs22, fs23, fs24, diss]:
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Normalise _id across FS files (FS_2024 may lack it)
for _df in [fs22, fs23, fs24]:
    if '_id' not in _df.columns:
        _df.insert(0, '_id', range(len(_df)))
print("FS schema normalised: _id column present in all three files")

print(f"  Company records  : {len(companies):>9,} rows | {companies.shape[1]} cols")
print(f"  FS 2022          : {len(fs22):>9,} rows")
print(f"  FS 2023          : {len(fs23):>9,} rows")
print(f"  FS 2024          : {len(fs24):>9,} rows")
print(f"  Dissolutions     : {len(diss):>9,} rows")

Loading raw data...
FS schema normalised: _id column present in all three files
  Company records  :   814,836 rows | 21 cols
  FS 2022          :   213,931 rows
  FS 2023          :   121,387 rows
  FS 2024          :    29,249 rows
  Dissolutions     :     3,291 rows


In [3]:
# Column rename map (handles different CRO export versions)
RENAME_MAP = {
    "company_reg_date":       "comp_reg_date",
    "registration_date":      "comp_reg_date",
    "company_dissolved_date": "comp_dissolved_date",
    "dissolved_date":         "comp_dissolved_date",
    "last_annual_return_date":"last_ar_date",
}
companies.rename(columns={k: v for k, v in RENAME_MAP.items() if k in companies.columns}, inplace=True)

REQUIRED = ["company_num","company_name","company_status",
            "comp_reg_date","comp_dissolved_date",
            "last_ar_date","last_accounts_date","nace_v2_code"]
missing_cols = [c for c in REQUIRED if c not in companies.columns]
if missing_cols:
    print(f"ERROR: missing required columns: {missing_cols}")
    raise ValueError(f"Required columns missing: {missing_cols}")
print(f"All {len(REQUIRED)} required columns present  ✅")
for c in REQUIRED:
    print(f"  {c}")

All 8 required columns present  ✅
  company_num
  company_name
  company_status
  comp_reg_date
  comp_dissolved_date
  last_ar_date
  last_accounts_date
  nace_v2_code


In [4]:
# Null rates and duplicates
dup_rows = companies.duplicated(subset="company_num").sum()
null_nace = companies["nace_v2_code"].isna().mean()
null_diss = companies["comp_dissolved_date"].isna().mean()
null_reg  = companies["comp_reg_date"].isna().mean()

print("Duplicates + null rates:")
print(f"  Duplicate company_num rows  : {dup_rows:,} (will be deduped in NB02)")
print(f"  nace_v2_code null rate      : {null_nace:.1%} (imputed in NB02 via keyword matching)")
print(f"  comp_dissolved_date null    : {null_diss:.1%} (correct — most companies are active)")
print(f"  comp_reg_date null          : {null_reg:.1%}")

# Status distribution
print()
print("Company status distribution (top 8):")
for status, count in companies["company_status"].value_counts().head(8).items():
    print(f"  {status:<35} {count:>9,}  ({count/len(companies):.1%})")

Duplicates + null rates:
  Duplicate company_num rows  : 375 (will be deduped in NB02)
  nace_v2_code null rate      : 69.0% (imputed in NB02 via keyword matching)
  comp_dissolved_date null    : 42.7% (correct — most companies are active)
  comp_reg_date null          : 2.0%

Company status distribution (top 8):
  Dissolved                             450,313  (55.3%)
  Normal                                326,886  (40.1%)
  Dissolved-20 years                     17,653  (2.2%)
  Liquidation                             9,207  (1.1%)
  Strike Off Listed                       3,088  (0.4%)
  Ceased IRL                              2,628  (0.3%)
  Dissolved PostMerger                    2,022  (0.2%)
  Ceased                                  1,254  (0.2%)


In [5]:
# Filing date ranges across FS files
print("Financial Statement filing date ranges:")
for name, df in [("FS 2022", fs22), ("FS 2023", fs23), ("FS 2024", fs24)]:
    if "submissions_accounts_to_date" in df.columns:
        dates = pd.to_datetime(df["submissions_accounts_to_date"], errors="coerce").dropna()
        print(f"  {name}: {dates.min().date()} → {dates.max().date()}  ({len(dates):,} valid)")
    else:
        print(f"  {name}: submissions_accounts_to_date column not found")

print()

# CRO submissions summary check
subs_path = RAW_FILES.get("cro_submissions_summary")
if subs_path and Path(subs_path).exists():
    size_mb = Path(subs_path).stat().st_size / 1024**2
    print(f"  cro_submissions_summary.csv  {size_mb:.1f} MB  ✅")
else:
    print("  WARNING: cro_submissions_summary.csv not found")

print()
print("Pre-flight PASSED — all files loaded, no blocking issues  ✅")
print("Ready for 02_feature_engineering_rebuilt.ipynb")

Financial Statement filing date ranges:
  FS 2022: 2022-01-01 → 2022-12-31  (213,931 valid)
  FS 2023: 2022-12-31 → 2024-12-31  (121,344 valid)
  FS 2024: 2024-01-31 → 2024-12-31  (29,158 valid)

  cro_submissions_summary.csv  116.6 MB  ✅

Pre-flight PASSED — all files loaded, no blocking issues  ✅
Ready for 02_feature_engineering_rebuilt.ipynb
